# preparation de l'environement 

dans le terminal `uv init` , ` uv venv `, ` Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process ` et ` .venv\Scripts\activate ` .  ` uv add ipykernel ` et  ` uv add langchain ` puis ` uv add langchain-openai ` et  `uv add langchain-text-splitters ` puis  `uv add langchain-community ` finalement pour importer la clee api open ai ` uv add python-dotenv  `


In [11]:
import json
import tiktoken
#import pandas as pd||
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader 
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
import os

# LLM

In [12]:
load_dotenv(override=True)

True

In [13]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o-mini",temperature=0)

# Implementation de RAG
   1. loading the pdf, Chunking

je vais créer un dossier que je vais appeler PDFs comme ça et à
l'intérieur duquel je vais coller ce fichier PDF nc346.pdf ( pur vous choisir votre cv par exemple ou un fichier que vous connaissez déjà ).

In [14]:
pdf_file="./pdfs/nc346.pdf"

In [15]:
pdf_loader=PyPDFLoader(pdf_file)

Et si vous voulez charger un dossier qui contient plusieurs fichiers, vous utilisez `PyPDFDirectoryLoader(path="./pdfs") `

On va utiliser split. Donc ici donc je rappelle ` RecursiveCharacterTextSplitter ` from tiktoken encoder. Voilà le modèle de Tiktoken utilisé. La taille de chaque chunk c'est 300 tokens et le overlap c'est 20.

Maintenant je vais charger le modèle TextSplitter, donc alors ici il y a une dépendance qui manque. Pour faire split, il vous demande d'installer ` pypdf `.

Donc  ` uv add pypdf `.

In [16]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

Dans le terminal de projet ajouter ` uv add pypdf `

Donc load_and_split, il va charger le fichier puis il va utiliser cette stratégie ici. Chaque chunk c'est 300 token et le overlap égal à 20. Donc si vous voulez voir le nombre de chunks ` len(chunks) `.

In [17]:
chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)


161

# 2.vectore Store - ChromaDB et Embedding

importer le model d'embedding 



In [18]:
from langchain_openai import OpenAIEmbeddings
embedding_model=OpenAIEmbeddings(model='text-embedding-ada-002')

Ajouter un dossier "store" et utiliser la commande ` uv add chromadb ` pour ajouter *chromadb*

 Je vais vectoriser les chunks et je les stock dans la base de données vectorielle ici **vectorstore**

In [19]:
vectorstore=Chroma.from_documents(chunks,embedding_model,collection_name="nc346_v2",persist_directory="./store")

# 2.2  
**Trouver les chunks liée à la questions (Retriever)**

In [25]:
#crée le Retriever 
retriever=vectorstore.as_retriever(search_type='similarity',search_Kwargs={'k',5})
# k=5. Donc je vais lui demander de me chercher les cinq chunks les plus proches à la question.


On va faire un test. J'ai la question:

QU'il est la capitalisation boursière de Maroc pour 2025 ? .
retriver.invoque il a cherché les chunks. Je vais les afficher

In [21]:
#test
retrieved_chunks=retriever.invoke(" QU'il est la capitalisation boursière de Maroc pour 2025 ?")
print(retrieved_chunks)
print("nombre de chunks est :",len(retrieved_chunks))


[Document(metadata={'total_pages': 52, 'moddate': '2025-12-26T11:38:19+01:00', 'creator': 'Adobe InDesign 20.5 (Macintosh)', 'page': 47, 'page_label': '48', 'trapped': '/False', 'source': './pdfs/nc346.pdf', 'creationdate': '2025-12-26T11:38:00+01:00', 'producer': 'Adobe PDF Library 17.0'}, page_content='48\nNote de Conjoncture \nDe son côté, la capitalisation boursière s’est \nrepliée, par rapport à fin octobre 2025, de 5,2% \npour se situer à 982,4 milliards de dirhams, \nramenant ainsi sa performance par rapport à \nfin décembre 2024 à +30,6% après +37,8% le \nmois dernier. La baisse mensuelle a résulté, \nparticulièrement, des contributions négatives \nnotables des secteurs des banques, du BTP , de \nMaroc Telecom, de l’électricité, des mines, de \nl’immobilier et de l’agroalimentaire.\nS’agissant du volume global des transactions réalisées au titre du mois de novembre 2025, il a reculé, \npar rapport au mois précédent, de 49,5% pour se situer à 5,1 milliards de dirhams. Ce volume 

Maintenant la partie rag, on va créer notre système prompt .

# 3. RAG et Q&A

-  *Prompt Design*

In [22]:
prompt_template="""
Answer the following question based only on provided context The context is about NOTE DE
CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES Décembre 2025
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

La note sur le contexte. Alors ça vous pouvez l'ajouter si vous savez que votre fichier document il concerne CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES. Vous lui dites le contexte il est à propos de NOTE DE CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES Décembre 2025. Donc ça c'est une phrase utile. Si vous voulez que ça soit général, quel que soit le fichier PDF, vous enlevez cette cette partie.

Le contexte est délimité par le tag contexte ` <context>`, la question est délimitée par le tag question ` <question> `. 

Voyez donc pour l'hallucination, regardez si la réponse n'existe pas avec ce context Répond avec Je ne sais pas! .
Donc ça c'est pour éliminer le problème de l'hallucination.

- *Retrieving the Relevant Documents*

passer à la partie retrieving.

Alors voyez donc je fais, je récupère les documents je l'ai converti. Pour chaque document (relevant_document_chunks), je vais prendre uniquement page content (d.page_content) et ça me donne une liste de textes. Et puis
cette liste de texte de chunks on va les concaténer par un séparateur ". " .


In [26]:
user_input="QU'il est la capitalisation boursière de Maroc pour 2025 ? "

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)


context_for_query

'48\nNote de Conjoncture \nDe son côté, la capitalisation boursière s’est \nrepliée, par rapport à fin octobre 2025, de 5,2% \npour se situer à 982,4 milliards de dirhams, \nramenant ainsi sa performance par rapport à \nfin décembre 2024 à +30,6% après +37,8% le \nmois dernier. La baisse mensuelle a résulté, \nparticulièrement, des contributions négatives \nnotables des secteurs des banques, du BTP , de \nMaroc Telecom, de l’électricité, des mines, de \nl’immobilier et de l’agroalimentaire.\nS’agissant du volume global des transactions réalisées au titre du mois de novembre 2025, il a reculé, \npar rapport au mois précédent, de 49,5% pour se situer à 5,1 milliards de dirhams. Ce volume est réparti \nà hauteur de :\n• 99,5% pour le marché central où les transactions ont reculé, par rapport au mois antérieur, de \n49,2% à 5 milliards de dirhams. Parmi les titres les plus actifs sur ce compartiment, figurent TGCC,. 48\nNote de Conjoncture \nDe son côté, la capitalisation boursière s’est \

In [29]:
len(relevant_document_chunks)

4

Maintenant, on va injecter dans le prompt le contexte et la question en utilisant template format. Vous voyez donc pour template On a injecté le contexte et la question. Et puis donc le si vous voulez afficher le prompt. Qu'est-ce qu'il contient ? Il contient:

 1. le système message:  ( Answer the following question ............  Je ne sais pas! `` <context> ``).
 2. le contexte:  entre ` <context> ` et `` </context> `` .
 3. question:  entre ` <question> ` et ` </question> ` .

Donc, ça c'est le prompt. Maintenant on va utiliser prompt. On va poser cette question là à llm.


In [30]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt)


Answer the following question based only on provided context The context is about NOTE DE
CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES Décembre 2025
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
48
Note de Conjoncture 
De son côté, la capitalisation boursière s’est 
repliée, par rapport à fin octobre 2025, de 5,2% 
pour se situer à 982,4 milliards de dirhams, 
ramenant ainsi sa performance par rapport à 
fin décembre 2024 à +30,6% après +37,8% le 
mois dernier. La baisse mensuelle a résulté, 
particulièrement, des contributions négatives 
notables des secteurs des banques, du BTP , de 
Maroc Telecom, de l’électricité, des mines, de 
l’immobilier et de l’agroalimentaire.
S’agissant du volume global des transactions réalisées au titre du mois de novembre 2025, il a reculé, 
par rapport au mois précédent, de 49,5% pour se situer à 5,1 milliards de di

- *récupérer la réponse de la part de llm*

In [32]:

resp=llm.invoke(prompt)
from IPython.display import Markdown
print(display(Markdown(resp.content)))

La capitalisation boursière de Maroc pour 2025 est de 982,4 milliards de dirhams.

None


On va définir rag comme fonction RAG.

On va lui donner query, llm et le prompt_template.

Donc le RAG, qu'est-ce qu'il fait ? Il fait d'abord retriever, on lui donne la question, il cherche les documents, il va les
convertir en une liste de textes. Chaque texte, ça représente page content. Et puis on va concaténer ce texte là par un séparateur. Ça devient le contexte. On injecte le contexte et la question dans le prompt template. On fait invoque et puis on on retourne le contenu.

 Vous voyez donc a la fonction qui nous intéresse. Pour pouvoir la tester à chaque fois si vous voulez lui poser une question vous faites appel RAG.


In [34]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)   
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

In [36]:
resp=RAG(" qu'ils sont les événements culturels dans le sud-est de Maroc ? ")
print(display(Markdown(resp)))

Je ne sais pas!

None


# 4.Evaluation 

![Logo](./rag_juge.gif)

In [39]:
#question ou query 
user_input=" QU'il est la capitalisation boursière de Maroc pour 2025 "

Récupérer le context 

In [40]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [41]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [ ]:
# Réponse d'un RAG
answer=RAG(" QU'il est la capitalisation boursière de Maroc pour 2025 ? ")
print(display(Markdown(answer)))

La capitalisation boursière de Maroc pour 2025 est de 982,4 milliards de dirhams.

None


- *Groundness*

Le prompt pour le juge 

In [43]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [ ]:
# définir le juge
groundness_checker=ChatOpenAI(
      model="gpt-4o",
      temperature=0
)

In [45]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question) 
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response=model.invoke(prompt)
    return juge_response.content



In [ ]:
#évaluer
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
print(display(Markdown(resp)))

**Étapes pour évaluer la réponse selon la métrique :**

1. **Identifier la question** : La question demande la capitalisation boursière du Maroc pour l'année 2025.

2. **Analyser le contexte** : Le contexte fournit des informations sur la capitalisation boursière à la fin d'octobre 2025, qui est de 982,4 milliards de dirhams. Il mentionne également une baisse de 5,2% par rapport à fin octobre 2025 et une performance de +30,6% par rapport à fin décembre 2024.

3. **Comparer la réponse avec le contexte** : La réponse indique que la capitalisation boursière pour 2025 est de 982,4 milliards de dirhams, ce qui correspond à l'information donnée dans le contexte pour la fin d'octobre 2025.

4. **Vérifier la précision et la pertinence** : La réponse est précise et pertinente par rapport au contexte fourni, car elle utilise directement l'information sur la capitalisation boursière à la fin d'octobre 2025.

**Explication étape par étape :**

- La question demande une valeur spécifique pour la capitalisation boursière du Maroc en 2025.
- Le contexte mentionne explicitement que la capitalisation boursière à la fin d'octobre 2025 est de 982,4 milliards de dirhams.
- La réponse fournie par l'IA reprend exactement cette information sans ajouter d'éléments externes ou non mentionnés dans le contexte.
- La réponse est donc directement dérivée des informations fournies dans le contexte.

**Évaluation de la métrique :**

La réponse respecte entièrement la métrique, car elle est directement dérivée des informations présentées dans le contexte. Elle ne contient aucune information supplémentaire ou incorrecte par rapport au contexte.

**Score : 5**

La réponse est entièrement conforme à la métrique, car elle utilise uniquement les informations fournies dans le contexte pour répondre à la question.

None
